In [2]:
import ee
import geemap

# Authenticate and initialize Earth Engine
# The user needs to authenticate and provide the project name when running this.
try:
    ee.Initialize(project="ee708-rainfall-downscaling")
except Exception as e:
    print(
        "Could not initialize Earth Engine with the project, trying to authenticate..."
    )
    ee.Authenticate()
    ee.Initialize(project="ee708-rainfall-downscaling")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


In [3]:
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")
era5 = ee.ImageCollection("ECMWF/ERA5/DAILY").select("total_precipitation")

In [ ]:
# Define the bounding box for the Himalayas region using specific coordinates.
himalayas_coords = [
    [76.47717699953913, 37.82664513320144],
    # [77.39147302732823, 36.47134074092161],
    # [79.67486846517245, 36.42314935342572],
    [80.90026755062496, 35.571803592889346],
    # [79.94743070144554, 32.78366923793975],
    # [81.87950044641099, 30.949760527513547],
    # [88.91295141369213, 28.71385417780965],
    # [94.50093992837051, 30.22256514409202],
    [97.21511065845843, 29.790425258748012],
    [97.9846584261799, 28.219530525561403],
    [97.17357167816135, 27.0017376993619],
    [93.32013533599073, 21.895638006891925],
    [92.12434489366078, 21.640209347541635],
    [91.44246786493959, 22.391938555703494],
    [87.81972158780012, 21.51648763575694],
    [87.17733874211916, 20.269295275574773],
    [80.95032103492291, 15.681782913430276],
    [80.28755447922079, 10.745717535241797],
    [80.03787822576705, 10.131854192880327],
    [77.5784805324031, 7.706238742849392],
    [76.24263442936031, 8.706332281101915],
    [74.43180260094009, 12.350805112400755],
    [72.39906488921469, 18.301660670307868],
    [67.94905459916471, 23.54293033675648],
    # [69.43726122140745, 27.764648638633588],
    [68.21883265960672, 23.247691821952614],
    [ 67.97708437455793, 23.839881647923207],
    [68.4698586761021, 24.46504974206444],
    [69.200023021483, 24.63005695508358],
    [70.58524083020568, 24.717456092256207],
    [69.16683481739948, 26.453152924415633],
    [69.00152321474258, 27.24672911668973],
    [69.91371433766922, 28.321561130479736],
    [71.14725193419356, 28.44732698104629],
    [73.27561368771326, 32.25007221718046],
    [71.84380570344365, 35.84156760093365],
    [73.14597692999519, 37.490491017813504],
    # [86.84464490510447, 35.64673507322577],  # Closing the polygon
]

region = ee.Geometry.Polygon(himalayas_coords)

dem_collection = ee.ImageCollection("COPERNICUS/DEM/GLO30").filterBounds(region)
dem = dem_collection.select('DEM').mosaic().clip(region)

# Define a new center point for map visualization based on the polygon's extent.
center_lon = 88.0
center_lat = 29.0

# Filter for the most recent image
most_recent_chirps = chirps.sort("system:time_start", False).first()
most_recent_era5 = era5.sort("system:time_start", False).first()
# most_recent_dem = dem.sort("system:time_start", False).first()

# Apply the mask to the image
precipitation_mask_chirps = most_recent_chirps.gt(0)
precipitation_mask_era5 = most_recent_era5.gt(0)
# dem_mask = dem.gt(0)

# Apply the mask to the image
precipitation_mask_chrips = most_recent_chirps.updateMask(precipitation_mask_chirps)
precipitation_mask_era5 = most_recent_era5.updateMask(precipitation_mask_chirps)
# dem_mask = dem.updateMask(dem_mask)

# Define visualization parameters for precipitation
vis_params_chirps = {
    "min": 0.0,
    "max": 90.0,
    "palette": ['#ffffff', '#00ffff', '#0080ff', '#da00ff', '#ffa400', '#ff0000']
}
vis_params_era5 = {
    "min": 0.0,
    "max": 0.2,
    "palette": ['#ffffff', '#00ffff', '#0080ff', '#da00ff', '#ffa400', '#ff0000'],
    # "palette": "palette"
}
vis_params_dem = {
    "min": 0.0,
    "max": 2000.0,
    "palette":['#ffffff', '#00ffff', '#0080ff', '#da00ff', '#ffa400', '#ff0000'],
    # "palette": "palette"
}

# Create a map centered on the region of interest
map_himalayas = geemap.Map(center=[center_lat, center_lon], zoom=5, basemap='HYBRID')

# Add the precipitation layer to the map, clipped to the region
map_himalayas.addLayer(precipitation_mask_chrips.clip(region), vis_params_chirps, "CHIRPS Precipitation")
map_himalayas.addLayer(precipitation_mask_era5.clip(region), vis_params_era5, "ERA5 Precipitation")
map_himalayas.addLayer(dem, vis_params_dem, "Copernicus DEM 30M")
# map_himalayas.to_html("./himalayas_map.html")
map_himalayas

Map(center=[29.0, 88.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …